# VEP-nAChR: Complete Project Documentation

**Variant Effect Predictor for Nicotinic Acetylcholine Receptors**

This notebook is a comprehensive guide to the VEP-nAChR project. It explains everything from the biological background to the code implementation, intended for anyone (mentors, colleagues, future students) who needs to understand what this project does and how.

**Table of Contents:**
1. Background: What is a Variant Effect Predictor?
2. Biology: nAChR Proteins and Mutations
3. The Dataset
4. Feature Engineering (the core of this project)
5. Machine Learning Pipeline
6. Comparison with VEP-ENaC (the reference project)
7. Results (with and without structural features)
8. Evaluation Metrics — What They Mean and What's a "Good" Score
9. How Features Are Converted to Numbers (The Math)
10. How to Run the Code
11. Project File Map
12. Next Steps / Roadmap

---

## 1. Background: What is a Variant Effect Predictor?

### 1.1 The Problem

Human DNA contains instructions for building proteins. Sometimes, a single letter in the DNA changes (a **mutation** or **variant**), which changes one amino acid in the resulting protein. This is called a **missense mutation**.

For example:
```
Normal protein:  ...M-E-L-K-G-A-T...
                       ^
Mutant protein:  ...M-A-L-K-G-A-T...
                       ^
                  Position 97: Glutamic acid (E) → Alanine (A)
```

This single amino acid change can have different effects:
- **Loss-of-Function (LOF):** The protein stops working or works less. Like breaking a switch so it can't turn on.
- **Gain-of-Function (GOF):** The protein becomes overactive or works differently. Like a switch stuck in the ON position.
- **No net effect:** The protein still works normally despite the change.

### 1.2 Why Predict This?

There are thousands of known mutations in nAChR genes, and new ones are discovered through genetic sequencing all the time. Testing each one in a lab (electrophysiology experiments) is expensive and slow. A computational predictor that can look at a mutation and say "this is likely LOF" or "this is likely GOF" based on the amino acid properties and protein structure would be extremely valuable for:
- **Clinical diagnosis:** A patient has a new mutation — is it pathogenic?
- **Drug development:** Understanding which mutations cause disease helps design targeted treatments.
- **Research prioritization:** Which of 500 unstudied mutations should we test in the lab first?

### 1.3 How a VEP Works (High Level)

```
Input: A mutation (e.g., "CHRNA7 E97A")
  ↓
Step 1: Convert the mutation into numerical features
        (physicochemical properties, structural context, etc.)
  ↓
Step 2: Feed the features into a trained ML model
  ↓
Output: Prediction → "LOF" or "GOF"
```

The hard part is Step 1: figuring out WHICH numerical features capture the information that determines whether a mutation causes LOF or GOF. This is called **feature engineering** and is the core of this project.

---

## 2. Biology: nAChR Proteins and Mutations

### 2.1 What are nAChRs?

**Nicotinic acetylcholine receptors (nAChRs)** are proteins that sit in the cell membrane of neurons and muscle cells. They form a channel (a pore) that allows ions (sodium, potassium, calcium) to flow across the membrane when activated.

**How they work:**
1. The neurotransmitter **acetylcholine (ACh)** binds to the receptor
2. This causes the channel to **open**
3. Ions flow through → electrical signal → muscle contracts or neuron fires
4. ACh unbinds → channel **closes**

They are called "nicotinic" because nicotine (from cigarettes) also activates them.

### 2.2 Protein Structure

Each nAChR is a **pentamer** — it is made of 5 subunit proteins assembled in a ring, with the ion channel pore in the center.

```
        Top view (looking down the pore):

            α ---- β
           / \    / \
          /   \  /   \
         δ    [PORE]   α      ← 5 subunits around a central pore
          \   /  \   /
           \ /    \ /
            β/ε --- γ
```

Each subunit has a characteristic structure:
- **Extracellular domain (ECD):** Sticks out of the cell. Contains the ACh binding site at the interface between subunits.
- **Transmembrane domain (TMD):** 4 alpha-helices (TM1-TM4) that span the cell membrane. TM2 lines the ion pore.
- **Intracellular domain (ICD):** A large loop between TM3 and TM4, inside the cell.

### 2.3 The 17 Human nAChR Genes

Humans have 17 genes encoding nAChR subunits. Different combinations form different receptor subtypes in different tissues:

| Gene | Subunit | Location | Receptor Type |
|------|---------|----------|---------------|
| CHRNA1 | α1 | Neuromuscular junction | Muscle-type (α1)₂β1δε |
| CHRNA2 | α2 | Brain | Neuronal heteromeric |
| CHRNA3 | α3 | Autonomic ganglia | α3β4 |
| CHRNA4 | α4 | Brain (widespread) | α4β2 (most common brain nAChR) |
| CHRNA5 | α5 | Brain | Accessory subunit in α4β2α5 |
| CHRNA6 | α6 | Brain (dopamine neurons) | α6β2β3 |
| CHRNA7 | α7 | Brain, immune cells | α7 homomeric (five α7 subunits) |
| CHRNA9 | α9 | Inner ear hair cells | α9α10 |
| CHRNA10 | α10 | Inner ear hair cells | α9α10 |
| CHRNB1 | β1 | Neuromuscular junction | Muscle-type |
| CHRNB2 | β2 | Brain (widespread) | α4β2 |
| CHRNB3 | β3 | Brain | Accessory subunit |
| CHRNB4 | β4 | Autonomic ganglia | α3β4 |
| CHRND | δ | Neuromuscular junction | Muscle-type |
| CHRNE | ε | Neuromuscular junction (adult) | Muscle-type (replaces γ after birth) |
| CHRNG | γ | Neuromuscular junction (fetal) | Fetal muscle-type |

### 2.4 Diseases Caused by nAChR Mutations

| Disease | Subunits | Effect | Symptoms |
|---------|----------|--------|----------|
| Congenital Myasthenic Syndrome (CMS) | CHRNA1, CHRNB1, CHRND, CHRNE | LOF (usually) | Muscle weakness, fatigue |
| Autosomal Dominant Nocturnal Frontal Lobe Epilepsy (ADNFLE) | CHRNA4, CHRNB2, CHRNA2 | GOF (usually) | Seizures during sleep |
| Nicotine dependence (risk factor) | CHRNA5, CHRNA3, CHRNB4 | Various | Increased addiction susceptibility |
| Multiple Pterygium Syndrome | CHRNG | LOF | Fetal akinesia, joint contractures |

### 2.5 LOF vs GOF — What Determines It?

Whether a mutation causes LOF or GOF depends on:
- **Where** in the protein the mutation is (binding site? pore-lining? intracellular?)
- **What** amino acid change occurs (conservative or radical?)
- **How** the change affects the protein's physical properties (charge, size, hydrophobicity)
- **The 3D structural context** (is the position buried or exposed? in a helix or loop?)

This is exactly what our features try to capture.

---

## 3. The Dataset

### 3.1 Data Source

The mutation data was manually curated from published research papers. Each entry represents a missense mutation that was experimentally tested (usually by electrophysiology) and classified as LOF or GOF.

### 3.2 Raw Data Format

The raw Excel file (`nachr_db_cleaned.xlsx`) has these columns:

| Column | Example | Description |
|--------|---------|-------------|
| OID | 1 | Row identifier |
| nAChR subunit | CHRNA7 | Which gene the mutation is in |
| Modification type | Substitution | Type of mutation (we only use substitutions) |
| AA position | 97 | Position in the protein sequence |
| Initial AA | E | Wildtype (original) amino acid |
| New AA | A | Mutant (replacement) amino acid |
| Effect | LOF | Experimentally determined effect |
| Measuring Technique | Electrophysiology | How the effect was measured |
| Pathology | CMS | Associated disease (if known) |
| Reference(PMID) | 33740418 | PubMed ID of the source paper |

### 3.3 Data Cleaning Steps

The raw database has 413 entries. We filter and clean as follows:

1. **Keep only substitutions:** Drop deletions (15), stop codons (10), frameshifts (4) → we need a WT→MT amino acid pair for our features
2. **Drop ambiguous labels:** Remove "LOF/GOF" entries (10) where the effect was unclear
3. **Drop no-net-effect:** Remove "No net effect" entries (~24) because there are too few for a third class (only ~5% of data)
4. **Fix whitespace:** Strip trailing spaces from "GOF " etc.
5. **Validate amino acids:** Ensure all WT and MT amino acids are one of the 20 standard single-letter codes

**Final dataset: 351 substitution mutations (218 LOF, 133 GOF)**

### 3.4 Mutation Distribution by Subunit

The data is heavily skewed — muscle-type subunits (especially CHRNA1 and CHRNE) dominate because Congenital Myasthenic Syndrome is the most-studied nAChR disease.

| Subunit | Count | % of total |
|---------|-------|------------|
| CHRNA1 | 104 | 29.6% |
| CHRNE | 65 | 18.5% |
| CHRNA7 | 40 | 11.4% |
| CHRND | 23 | 6.6% |
| CHRNA4 | 21 | 6.0% |
| CHRNA6 | 18 | 5.1% |
| CHRNB1 | 18 | 5.1% |
| CHRNB2 | 17 | 4.8% |
| CHRNB4 | 15 | 4.3% |
| CHRNA2 | 9 | 2.6% |
| CHRNA3 | 7 | 2.0% |
| CHRNA5 | 5 | 1.4% |
| CHRNB3 | 4 | 1.1% |
| CHRNG | 3 | 0.9% |
| CHRNA9 | 2 | 0.6% |

### 3.5 FASTA Sequence Files

For each subunit, we have the complete amino acid sequence downloaded from NCBI as FASTA files. These are used for:
- Validating that the WT amino acid in the database matches the actual sequence
- Aligning to PDB structures (for structural features)
- Future MSA conservation analysis

Each FASTA file contains multiple isoforms. We pick the **canonical isoform** (isoform 1 / primary RefSeq entry) for each subunit.

---

## 4. Feature Engineering (The Core of This Project)

Feature engineering is the process of converting raw mutation information ("CHRNA7 E97A") into a vector of numbers that a machine learning model can learn from. This is the most important part of the project — the choice of features determines how well the model can distinguish LOF from GOF.

### 4.0 The Big Picture

Each mutation is represented as **44 numbers** (without structural features) or **50 numbers** (with structural features). These numbers capture different aspects of what the mutation "looks like" to the protein:

```
Input:  CHRNA7 E97A (LOF)

                    ┌─ WT properties (8 numbers): What is Glutamic acid like?
                    │  MT properties (8 numbers): What is Alanine like?
  Physicochemical ──┤  Differences  (8 numbers): How different are they?
  (24 features)     │
                    └─ e.g., E is charged, A is neutral → diff_charge = +0.5

                    ┌─ BLOSUM62 score: Is E→A common in evolution? (no, score = -1)
  Substitution ─────┤  BLOSUM62 normalized: Same, scaled to [0,1]
  (3 features)      └─ Grantham distance: How physically different? (107/215 = moderate)

                    ┌─ Position normalized: Where in the protein? (97/1314 = 0.074, near N-terminus)
  Positional ───────┤
  (17 features)     └─ Subunit one-hot: Which gene? [0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0] = CHRNA7

                    ┌─ RSA: Is position 97 buried or exposed?
                    │  B-factor: Is it rigid or flexible?
  Structural ───────┤  DSSP: Is it in a helix, sheet, or loop?
  (6 features)      │  C-beta density: How tightly packed is this region?
                    └─ (currently imputed — needs PDB files)

Output: [0.458, 0.914, 0.426, ..., 1.0, 0.0, ..., 0.0]  ← 44 numbers total
```

### 4.1 Physicochemical Features (24 features)

**File:** `vep_nachr/features/physicochemical.py`  
**From ENaC:** `VEP-Enac/vep/features/engineered.py` — taken with no changes.

#### What are amino acid properties?

There are 20 standard amino acids, each with different physical and chemical properties. These properties determine how the amino acid behaves in a protein. The **AAIndex** database (https://www.genome.jp/aaindex/) is a collection of hundreds of published scales that quantify these properties.

We use 8 carefully chosen scales:

#### Property 1: Hydrophobicity (EISD840101)
- **What:** How much does this amino acid avoid water?
- **Range:** Leucine (L) is the most hydrophobic (1.0), Arginine (R) is the least (0.0)
- **Why it matters:** Proteins fold so that hydrophobic residues are buried inside (away from water) and hydrophilic residues are on the surface. If you replace a buried hydrophobic residue with a hydrophilic one, the protein may misfold → LOF.
- **Example:** L→R (hydrophobic→hydrophilic) at a buried position = almost certainly damaging

#### Property 2: Polarity (GRAR740102)
- **What:** How unevenly distributed is the electron cloud?
- **Range:** Aspartate (D) is most polar (1.0), Leucine (L) is least (0.0)
- **Why it matters:** Polar residues form hydrogen bonds. Losing or gaining polarity can break critical interactions.

#### Property 3: Volume (KRIW790103)
- **What:** How physically large is the side chain?
- **Range:** Tryptophan (W) is largest (1.0), Glycine (G) is smallest (0.0)
- **Why it matters:** Putting a large amino acid where a small one was (or vice versa) causes steric clashes or creates cavities. In tightly packed regions, even small size changes can be disruptive.
- **Example:** G→W (tiny→huge) in the channel pore would block ion flow.

#### Property 4: Molecular Weight (FASG760101)
- **What:** The mass of the amino acid in Daltons.
- **Why it matters:** Correlated with volume but captures additional chemistry (heavier atoms, more complex side chains).

#### Property 5: Charge at pH 7 (KLEP840101)
- **What:** Net electric charge at physiological pH.
- **Key values:** D, E = negative (0.0). K, R = positive (1.0). All others = neutral (0.5).
- **Why it matters:** Charged residues form salt bridges (electrostatic bonds). Losing a charge can break these critical structural contacts. Also, the nAChR ion pore has rings of charged residues that control ion selectivity — mutations here are often GOF or LOF.
- **Example:** E→A at the pore lining removes a negative charge → changes ion conductance.

#### Property 6: Isoelectric Point (ZIMJ680104)
- **What:** The pH at which the amino acid has zero net charge.
- **Why it matters:** Gives a more nuanced view of charge behavior across different pH environments.

#### Property 7: Aromaticity (binary)
- **What:** Does the side chain contain an aromatic ring?
- **Values:** F (Phenylalanine), W (Tryptophan), Y (Tyrosine), H (Histidine) = 1. All others = 0.
- **Why it matters:** Aromatic residues participate in pi-stacking interactions and are important for ligand binding. In nAChRs, aromatic residues in the binding pocket form the "aromatic box" that coordinates acetylcholine.

#### Property 8: Secondary Structure Preference (CHOP780201)
- **What:** The tendency of this amino acid to be found in alpha-helices (Chou & Fasman scale).
- **Range:** Glutamate (E) has the highest helix preference (1.0), Glycine (G) the lowest (0.0).
- **Why it matters:** The TM domains of nAChRs are alpha-helices. Introducing a helix-breaking residue (like Proline) into TM2 would disrupt the pore structure → likely LOF.

#### How we compute them

For each mutation, we compute **three sets** of values:

1. **WT properties** (8 values): Look up the wildtype amino acid in the table
2. **MT properties** (8 values): Look up the mutant amino acid in the table
3. **Differences** (8 values): MT - WT for each property

The differences are the most informative — they tell the model "how much did this property change?" A large diff_charge means the mutation dramatically altered the electrostatics. A large diff_hydrophobicity means a polar↔hydrophobic swap.

All values are **min-max normalized to [0, 1]** across the 20 standard amino acids, so different scales are comparable.

#### Example: CHRNA7 E97A

```
Property                    WT (E)   MT (A)   Diff (A-E)
─────────────────────────   ──────   ──────   ─────────
hydrophobicity               0.458    0.807    +0.349  ← A is more hydrophobic
polarity                     0.914    0.395    -0.519  ← A is much less polar
volume                       0.426    0.189    -0.237  ← A is smaller
molecular_weight             0.558    0.109    -0.449  ← A is lighter
charge                       0.000    0.500    +0.500  ← E is negative, A is neutral!
isoelectric_point            0.056    0.404    +0.348
aromaticity                  0.000    0.000    +0.000  ← neither has a ring
secondary_structure_pref     1.000    0.904    -0.096  ← both like helices
```

The model sees that this mutation removes a negative charge (diff_charge = +0.5) and shifts from polar to hydrophobic (diff_polarity = -0.52). These are significant changes that could disrupt a salt bridge → LOF.

### 4.2 Substitution Scores (3 features)

**File:** `vep_nachr/features/substitution.py`  
**From ENaC:** BLOSUM62 part taken from ENaC. Grantham distance is NEW (ENaC did not have it).

#### BLOSUM62 — Evolutionary Substitution Score

**What is it?** BLOSUM62 (BLOcks SUbstitution Matrix) is a 20×20 matrix that scores every possible amino acid pair. It was built by analyzing real protein evolution — comparing related proteins and counting how often each substitution actually occurs.

- **High score** (e.g., D→E = +2): These amino acids are frequently swapped in evolution. They have similar properties, so the swap is tolerated. "Conservative substitution."
- **Low score** (e.g., C→W = -4): This swap is almost never seen in evolution. The amino acids are very different. "Radical substitution."
- **Diagonal** (e.g., W→W = +15): No change. Maximum score.

**Why it's useful:** If evolution has "tested" this substitution and found it acceptable (high BLOSUM score), the mutation is probably less damaging. If evolution has avoided it (low BLOSUM score), it's probably disruptive.

We store:
- `blosum62_raw`: The raw integer score (-4 to 11 for substitutions)
- `blosum62_normalized`: Scaled to [0, 1] for the model

#### Grantham Distance — Physicochemical Distance (NEW)

**What is it?** A composite distance metric published by Grantham in 1974. It combines three physicochemical properties (composition, polarity, molecular volume) into a single number that represents how "different" two amino acids are.

- **Range:** 0 (identical amino acids) to 215 (Cysteine → Tryptophan, the most different pair)
- **Low distance** (e.g., I→L = 5): Very similar amino acids. Conservative change.
- **High distance** (e.g., C→W = 215): Extremely different. Radical change.

**Why we added it (ENaC didn't have it):**
BLOSUM62 is based on **statistical observation** — how often does this substitution happen in evolution? Grantham is based on **physicochemical theory** — how different are these amino acids in terms of measurable properties? They capture complementary information. A substitution might be rare in evolution (low BLOSUM) not because it's damaging, but because the codon change requires multiple nucleotide mutations. Grantham cuts through this by looking only at the physical difference.

We normalize to [0, 1] by dividing by 215 (the max).

#### Example: E→A
```
BLOSUM62 raw:    -1   (moderately penalized — not a common evolutionary swap)
BLOSUM62 norm:   0.20 (on a [0,1] scale)
Grantham raw:    107  (moderate physicochemical distance)
Grantham norm:   0.50 (on a [0,1] scale)
```

### 4.3 Positional and Subunit Features (17 features)

**File:** `vep_nachr/features/encoder.py`  
**From ENaC:** Adapted (16 subunits instead of 3, no species encoding).

#### Normalized Position (1 feature)

The amino acid position of the mutation, divided by the maximum position in the dataset.

```
position_normalized = mutation_position / 1314
```

(1314 is the longest sequence position in our dataset)

**Why it matters:** nAChR subunits have a defined domain structure:
- Positions ~1-20: Signal peptide (cleaved off, no mutations here)
- Positions ~20-210: Extracellular domain (ACh binding site)
- Positions ~210-430: Transmembrane domains (TM1-TM4, the pore)
- Positions ~310-450: Intracellular loop (between TM3-TM4)
- Positions ~430-500+: C-terminal extracellular region

So the position tells the model roughly which domain the mutation is in. Mutations in the transmembrane domain (especially TM2, which lines the pore) tend to have more severe effects.

#### Subunit One-Hot Encoding (16 features)

Each of the 16 nAChR genes gets a binary column. For a mutation in CHRNA7, the CHRNA7 column is 1 and all others are 0.

**Why it matters:** Different subunits have different roles, expression patterns, and mutation sensitivities:
- CHRNA1 mutations are mostly LOF (muscle CMS)
- CHRNA4 mutations are often GOF (epilepsy)
- The model needs to know which subunit to adjust its predictions

**ENaC had 3 subunit columns** (alpha, beta, gamma). We have 16 because nAChR has many more genes.

### 4.4 Structural Features (6 features — NOW IMPLEMENTED)

**File:** `vep_nachr/features/structural.py`  
**From ENaC:** `VEP-Enac/vep/features/noah_features/structural_features.py` — adapted for multi-PDB support.

**Status: WORKING** — B-factor, DSSP, and C-beta density are extracted from PDB structures. RSA requires the `mkdssp` binary (not available on Windows) and is currently imputed as 1.0.

#### How It Works (Step by Step)

1. **Load the CIF structure file** using BioPython's MMCIFParser
2. **Align the UniProt sequence to the PDB chain** using BLOSUM62 global alignment (because PDB residue numbering doesn't always match UniProt numbering — there can be missing loops, extra residues, etc.)
3. **For each mutation**, look up which PDB residue corresponds to the UniProt position
4. **Extract features** from that residue's 3D coordinates

#### PDB Chain Assignments (determined by sequence alignment)

| PDB ID | Chain | Subunit | Mutations Covered |
|--------|-------|---------|-------------------|
| 7QKO | A | CHRNA1 | 104 |
| 7QKO | B | CHRNB1 | 18 |
| 7QKO | C | CHRND | 23 |
| 7QKO | D | CHRNE | 65 |
| 7QKO | E | CHRNG | 3 |
| 7EKI | A | CHRNA7 | 40 |
| 6CNJ | A | CHRNA4 | 21 |
| 6CNJ | B | CHRNB2 | 17 |
| 6PV7 | A | CHRNA3 | 7 |
| 6PV7 | B | CHRNB4 | 15 |

**Coverage: 267/351 mutations (76%) mapped to PDB, 84/351 imputed**

The 84 imputed mutations are from subunits without experimental structures (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3) or positions that fall outside the resolved structure.

#### Feature Extraction Results

| Feature | Min | Max | Mean | Status |
|---------|-----|-----|------|--------|
| rsa | 1.0 | 1.0 | 1.0 | Imputed (needs mkdssp binary) |
| bfactor | 0.0 | 211.0 | 71.6 | **Working** — real data from PDB |
| dssp_helix | 0 | 1 | 0.28 | **Working** — from mmCIF annotations |
| dssp_sheet | 0 | 1 | 0.17 | **Working** — from mmCIF annotations |
| dssp_coil | 0 | 1 | 0.56 | **Working** — from mmCIF annotations |
| cbeta_density | 0 | 26 | 11.7 | **Working** — from 3D coordinates |

#### What are PDB/CIF Files?

A **PDB/CIF file** contains the 3D coordinates (x, y, z) of every atom in a protein, determined by experiments like X-ray crystallography or cryo-electron microscopy. CIF (Crystallographic Information File) is the modern format replacing the older PDB format.

#### Feature: RSA (Relative Solvent Accessibility)

- **What:** What fraction of this residue's surface is exposed to water (solvent)?
- **Range:** 0.0 (completely buried inside the protein) to 1.0 (fully exposed on the surface)
- **How it's computed:** The DSSP algorithm calculates the solvent-accessible surface area (ASA) for each residue. We divide by the maximum possible ASA for that amino acid type:
  ```
  RSA = ASA / MaxASA[amino_acid_type]
  ```
- **Currently imputed as 1.0** because the `mkdssp` binary is not available on Windows. To get real RSA values, install mkdssp (Linux/Mac) or use a pre-computed DSSP file.
- **Why it matters:** Buried residues (RSA < 0.2) are in the protein core. Mutations here are more likely to be damaging.

#### Feature: B-factor (Temperature Factor)

- **What:** How much does this residue move/vibrate in the structure?
- **Unit:** Angstroms squared (Å²)
- **How it's computed:** Average B-factor of all heavy (non-hydrogen) atoms in the residue:
  ```
  B_factor = mean([atom.bfactor for atom in residue if atom.element != 'H'])
  ```
- **Fallback chain:** If the residue is missing, try adjacent residues, then chain median
- **Why it matters:** Low B-factor = rigid, structurally important. High B-factor = flexible, often in loops.

#### Feature: DSSP Secondary Structure (3 binary features)

- **What:** Is this residue in an alpha-helix, beta-sheet, or coil/loop?
- **How it's computed:** Parsed from mmCIF secondary structure annotations:
  - Helix annotations (HELX_*) → `dssp_helix = 1`
  - Sheet annotations (SHEET) → `dssp_sheet = 1`
  - Everything else → `dssp_coil = 1`
- **Why it matters:** In nAChRs, the transmembrane helices (TM1-TM4) are critical. A mutation that breaks a helix is likely damaging.

#### Feature: C-beta Density

- **What:** How many amino acids are packed around this position within a 10 Angstrom radius?
- **How it's computed:**
  1. Collect all C-beta atom coordinates in the chain (C-alpha for Glycine)
  2. Build a KDTree (spatial search structure) for fast neighbor queries
  3. Count neighbors within 10Å, subtract 1 (to exclude self)
- **Why it matters:** High density = tightly packed core. Mutations here are more disruptive.

#### What is AlphaFold?

**AlphaFold** is an AI system by Google DeepMind that predicts a protein's 3D structure from its amino acid sequence alone. For subunits without experimental structures (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3), AlphaFold structures can be downloaded from https://alphafold.ebi.ac.uk/ by searching the UniProt ID.

### 4.5 Feature Summary Table

| Index | Feature Name | Group | Range | What It Captures |
|-------|-------------|-------|-------|------------------|
| 0-7 | wt_{property} | Physicochemical | [0, 1] | Properties of the original amino acid |
| 8-15 | mt_{property} | Physicochemical | [0, 1] | Properties of the replacement amino acid |
| 16-23 | diff_{property} | Physicochemical | [-1, 1] | How much each property changed |
| 24 | blosum62_raw | Substitution | [-4, 11] | Evolutionary likelihood of this swap |
| 25 | blosum62_normalized | Substitution | [0, 1] | Same, scaled |
| 26 | grantham_normalized | Substitution | [0, 1] | Physicochemical distance between AAs |
| 27 | position_normalized | Positional | [0, 1] | Where in the protein (N-term to C-term) |
| 28-43 | subunit_{name} | Positional | {0, 1} | Which nAChR gene (one-hot) |
| 44 | rsa | Structural | [0, 1] | Solvent exposure (buried vs surface) |
| 45 | bfactor | Structural | [0, ∞) | Atomic mobility (rigid vs flexible) |
| 46 | dssp_helix | Structural | {0, 1} | In an alpha-helix? |
| 47 | dssp_sheet | Structural | {0, 1} | In a beta-sheet? |
| 48 | dssp_coil | Structural | {0, 1} | In a loop/coil? |
| 49 | cbeta_density | Structural | [0, ~30] | Packing density (how crowded) |

**Total: 44 features** (working) **→ 50 features** (when structural is implemented)

---

## 5. Machine Learning Pipeline

### 5.1 The Classification Task

Given a 44-dimensional feature vector representing a mutation, predict whether it causes **LOF (0)** or **GOF (1)**.

This is a **binary classification** problem with **imbalanced classes** (62% LOF vs 38% GOF).

### 5.2 Why We Use Multiple Models

No single ML algorithm is best for all problems (this is called the "No Free Lunch" theorem). Different models make different assumptions:

| Model | How It Works (Simple Explanation) | Strengths | Weaknesses |
|-------|----------------------------------|-----------|------------|
| **Logistic Regression** | Draws a straight line (hyperplane) to separate LOF and GOF in feature space | Interpretable, fast, good baseline | Can't capture non-linear patterns |
| **SVM (RBF)** | Finds the best separating boundary, but can curve it using the "kernel trick" | Good with small data, handles non-linearity | Sensitive to feature scaling, slow to tune |
| **Random Forest** | Builds 100+ decision trees on random subsets, takes majority vote | Handles non-linearity, robust, gives feature importance | Can overfit with too many trees |
| **LightGBM / XGBoost** | Builds trees sequentially, each one correcting the errors of the previous | Often best for tabular data, fast | Can overfit, harder to interpret |
| **KNN** | Looks at the K most similar mutations in the training set, takes majority vote | Simple, no assumptions | Slow at prediction time, sensitive to irrelevant features |
| **MLP** | A small neural network with one hidden layer | Can learn any function | Needs lots of data (we don't have enough) |
| **Gaussian NB** | Assumes each feature independently follows a bell curve per class | Very fast, works with small data | Strong independence assumption rarely holds |

### 5.3 Handling Class Imbalance

We have 218 LOF vs 133 GOF (62% vs 38%). If the model just predicts LOF for everything, it gets 62% accuracy "for free." To prevent this:

- **`class_weight='balanced'`**: Most models support this. It tells the model to penalize misclassifying GOF samples more heavily, proportional to how rare they are. Internally, it multiplies the loss for GOF samples by (218/133 ≈ 1.64).
- **Stratified splitting**: When dividing data into train/test folds, we ensure each fold has the same LOF/GOF ratio as the full dataset.

### 5.4 Evaluation Metrics

#### F1 Score (Primary Metric)

F1 is the harmonic mean of precision and recall for the GOF class:

```
Precision = True GOF predictions / All GOF predictions
            "When the model says GOF, how often is it right?"

Recall    = True GOF predictions / All actual GOF mutations
            "Of all real GOF mutations, how many does the model catch?"

F1        = 2 × (Precision × Recall) / (Precision + Recall)
            Balances both — penalizes if either is low
```

F1 = 0.654 means the model is moderately good at finding GOF mutations but still misses some.

#### Accuracy (Secondary Metric)

```
Accuracy = Correct predictions / Total predictions
```

Simple but misleading with imbalanced data. A "predict all LOF" baseline gets 62%.

#### Confusion Matrix

```
                    Predicted LOF    Predicted GOF
Actual LOF              TN               FP
Actual GOF              FN               TP

TN = True Negative:  Correctly predicted LOF
FP = False Positive: Predicted GOF but was actually LOF
FN = False Negative: Predicted LOF but was actually GOF
TP = True Positive:  Correctly predicted GOF
```

### 5.5 Cross-Validation: How We Split the Data

We NEVER train and evaluate on the same data. That would be like letting a student see the exam answers before taking the test — the score would be meaninglessly high.

#### Quick Mode (`--quick`): Simple 5-Fold CV × 5 Seeds

```
351 mutations
  ├── Fold 1: [70 test] [281 train]  ← train model, test on held-out 70
  ├── Fold 2: [70 test] [281 train]  ← different 70 held out
  ├── Fold 3: [70 test] [281 train]
  ├── Fold 4: [70 test] [281 train]
  └── Fold 5: [71 test] [280 train]
                                      ← every mutation gets tested exactly once

This is repeated 5 times with different random shuffles (seeds).
Total: 5 folds × 5 seeds = 25 evaluations → report mean ± std
```

**Split ratio: 80% train / 20% test** (rotating so all data is tested)

No hyperparameter tuning. Uses default model settings.

#### Full Mode: Nested CV (5 Outer × 5 Inner × 5 Seeds)

```
351 mutations
  ├── Outer Fold 1: [70 TEST] [281 for training]
  │     └── Inner optimization on the 281:
  │           ├── Inner Fold 1: [56 valid] [225 train] ← try HP set #1, score = 0.62
  │           ├── Inner Fold 2: [56 valid] [225 train] ← try HP set #1, score = 0.65
  │           ├── ... (50 Optuna trials × 5 inner folds)
  │           └── Best HPs found → train on all 281 → predict on 70 TEST
  ├── Outer Fold 2: [70 TEST] [281 for training]
  │     └── (same inner optimization)
  └── ...
```

**Split ratio: 64% train / 16% validation (inner) / 20% test (outer)**

This is the gold standard. The test set is NEVER seen during hyperparameter tuning, so the reported score is an honest estimate of real-world performance.

### 5.6 Hyperparameter Optimization (Optuna)

Every ML model has settings (hyperparameters) that affect how it learns:
- Random Forest: how many trees? how deep can each tree be?
- SVM: how much regularization? how flexible is the boundary?
- LightGBM: learning rate? number of leaves?

**Optuna** is a library that automatically searches for the best hyperparameters. It uses a smart search strategy called TPE (Tree-structured Parzen Estimator) that learns from previous trials to focus on promising regions of the search space.

We run 50 Optuna trials per fold. Each trial:
1. Suggests a set of hyperparameters
2. Trains the model with those HPs on the inner training set
3. Evaluates on the inner validation set
4. Reports the F1 score back to Optuna

After 50 trials, we take the best-performing HP set and use it for the outer fold evaluation.

### 5.7 Feature Scaling

Some models (Logistic Regression, SVM, KNN, MLP) are sensitive to feature scales. If one feature ranges from 0-1 and another from 0-1000, the model will be dominated by the larger one.

We use **RobustScaler** (from sklearn) which scales each feature by its interquartile range. This is more robust to outliers than standard scaling (mean/std).

Tree-based models (Random Forest, LightGBM, XGBoost) are **not** affected by feature scales — they make split decisions that are scale-invariant.

---

## 6. Comparison with VEP-ENaC (The Reference Project)

This project is adapted from VEP-ENaC, a Variant Effect Predictor for the Epithelial Sodium Channel (ENaC) built by a senior colleague.

### 6.1 The Proteins

| Aspect | ENaC | nAChR |
|--------|------|-------|
| **Full name** | Epithelial Sodium Channel | Nicotinic Acetylcholine Receptor |
| **Ion channel type** | Constitutively open Na+ channel | Ligand-gated cation channel |
| **Oligomeric state** | Trimer (3 subunits) | Pentamer (5 subunits) |
| **Subunit genes** | 3 (α, β, γ) | 17 (CHRNA1-10, CHRNB1-4, CHRND, CHRNE, CHRNG) |
| **Superfamily** | DEG/ENaC | Cys-loop receptors |
| **Gating** | Always open, regulated by proteases | Opens when acetylcholine binds |

Both are ion channels, so the feature engineering approach (physicochemical properties, structural context, substitution scores) is applicable to both. The biology is different enough that the exact weights the model learns will differ, but the feature types are the same.

### 6.2 Code Reuse Summary

| Component | Our File | ENaC Source | Changes Made |
|-----------|----------|-------------|-------------|
| Physicochemical features | `features/physicochemical.py` | `features/engineered.py` | **None** — amino acid properties are universal |
| BLOSUM62 scores | `features/substitution.py` | `features/engineered.py` | **Minor** — extracted into own module |
| Grantham distance | `features/substitution.py` | N/A | **New** — ENaC didn't have this |
| Structural features | `features/structural.py` | `features/noah_features/structural_features.py` | **Skeleton** — needs adaptation for multi-PDB |
| Feature encoder | `features/encoder.py` | `features/engineered.py` | **Adapted** — 16 subunits instead of 3, no species |
| Model registry | `models/registry.py` | `models/registry.py` | **Adapted** — binary scoring, removed CatBoost |
| Cross-validation | `training/cross_validation.py` | `training/cross_validation.py` | **Simplified** — removed species transfer, binary F1 |
| Data loader | `data/loader.py` | `data/` (multiple files) | **Rewritten** — Excel format, different columns |
| Config | `config.py` | `config.py` | **Rewritten** — nAChR subunits, binary labels |

### 6.3 What We Did NOT Take from ENaC

| ENaC Component | What It Was | Why We Skipped It |
|----------------|-------------|------------------|
| Noah's Original Features | A second feature pipeline with different AAIndex properties | Redundant — we consolidated into one cleaner pipeline |
| Data-Driven Encoders | Ordinal, one-hot, full-sequence encoding alternatives | Engineered features performed best in ENaC |
| Species Transfer Experiment | Train on human vs mouse vs both | We only have human data (mouse coming later) |
| Ablation Studies | Remove one feature group at a time to measure impact | Will add once baseline is solid |
| CatBoost model | Yandex gradient boosting library | Sklearn compatibility issues; XGBoost/LightGBM sufficient |
| Paper/figure scripts | LaTeX, matplotlib publication figures | Not needed yet |

### 6.4 What We Added That ENaC Didn't Have

| Addition | Description |
|----------|-------------|
| Grantham distance | Physicochemical distance score (complementary to BLOSUM62) |
| 16-way subunit encoding | Much more subunit resolution (ENaC had only 3) |

---

## 7. Results

### 7.1 Baseline (Quick Mode, No HP Tuning, No Structural Features)

351 samples (218 LOF, 133 GOF), 44 features, 5-fold CV x 5 seeds:

| Model | F1 (GOF) | +/- Std | Accuracy | +/- Std | Notes |
|-------|----------|---------|----------|---------|-------|
| **XGBoost** | **0.655** | 0.070 | **0.744** | 0.051 | Best F1 |
| **LightGBM** | **0.654** | 0.068 | 0.737 | 0.051 | Very close to XGBoost |
| KNN | 0.654 | 0.090 | 0.746 | 0.046 | Best accuracy, high variance |
| Logistic Regression | 0.648 | 0.067 | 0.706 | 0.046 | Solid linear baseline |
| Random Forest | 0.635 | 0.079 | 0.737 | 0.057 | Expected to improve with HP tuning |
| SVM (RBF) | 0.600 | 0.069 | 0.664 | 0.051 | Needs HP tuning |
| Gaussian NB | 0.588 | 0.028 | 0.502 | 0.066 | Bad accuracy (near random) |
| MLP | 0.422 | 0.126 | 0.663 | 0.058 | Unstable, not enough data |

### 7.2 With Structural Features (Quick Mode, No HP Tuning)

351 samples (218 LOF, 133 GOF), **50 features**, 5-fold CV x 5 seeds:

| Model | F1 (w/ struct) | F1 (w/o struct) | Change | Accuracy |
|-------|----------------|-----------------|--------|----------|
| **XGBoost** | **0.666** | 0.655 | +0.011 | 0.755 |
| **LightGBM** | **0.666** | 0.654 | +0.012 | 0.748 |
| Logistic Regression | 0.653 | 0.648 | +0.005 | 0.717 |
| Random Forest | 0.627 | 0.635 | -0.008 | 0.743 |

Structural features provide a **modest improvement** (+1-2% F1 for gradient boosting models). Note: RSA is still imputed (mkdssp unavailable on Windows), so once that's fixed, the improvement could be larger.

### 7.3 Comparison with ENaC

ENaC's best result was F1 ~ 0.538 (macro, 3-class). Our best is F1 ~ 0.666 (binary). This isn't an apples-to-apples comparison because:
1. Binary classification (2 classes) is inherently easier than 3-class
2. F1_binary and F1_macro are computed differently
3. Different proteins, different data distributions

But it's encouraging that the same feature engineering approach works for a different protein family.

---

## 10. How to Run the Code

### 10.1 Setup

```bash
cd "VEP Nachr"
pip install -r requirements.txt
```

### 10.2 Quick Test (Default Hyperparameters, No Optuna)

```bash
# Without structural features (fast, no PDB needed):
python scripts/run_experiment.py --quick --no-structural

# With structural features (requires CIF files in data/raw/structure_files/):
python scripts/run_experiment.py --quick
```

### 10.3 Full Experiment (With Optuna HP Optimization)

```bash
python scripts/run_experiment.py
```

This runs nested cross-validation with 50 Optuna trials per fold. Much slower but gives more accurate performance estimates.

### 10.4 Single Model

```bash
python scripts/run_experiment.py --model random_forest
```

### 10.5 All Models (Core + Extended)

```bash
python scripts/run_experiment.py --all-models --quick
```

### 10.6 Flags

| Flag | What it does |
|------|-------------|
| `--quick` | Use default HPs, skip Optuna optimization |
| `--no-structural` | Exclude structural features (use when no PDB files available) |
| `--model NAME` | Run only one model (e.g., `random_forest`, `lightgbm`) |
| `--all-models` | Run all 9 models instead of just the 4 core ones |
| `--n-trials N` | Number of Optuna trials per fold (default: 50) |

---

## 11. Project File Map

```
VEP Nachr/
│
├── NOTES.ipynb                              # This documentation file
├── requirements.txt                         # Python dependencies (pip install -r requirements.txt)
│
├── scripts/
│   └── run_experiment.py                    # Main entry point for running experiments
│
├── vep_nachr/                               # The Python package (all source code)
│   ├── __init__.py                          # Package marker, version info
│   ├── config.py                            # All settings: paths, subunits, PDB mappings, labels
│   │
│   ├── data/
│   │   ├── __init__.py
│   │   └── loader.py                        # Load Excel DB, clean data, load FASTA sequences
│   │
│   ├── features/
│   │   ├── __init__.py
│   │   ├── physicochemical.py               # 24 AAIndex features (from ENaC, unchanged)
│   │   ├── substitution.py                  # BLOSUM62 (from ENaC) + Grantham distance (new)
│   │   ├── structural.py                    # PDB-based features (WORKING for B-factor, DSSP, C-beta)
│   │   └── encoder.py                       # Combines all features into sklearn transformer
│   │
│   ├── models/
│   │   ├── __init__.py
│   │   └── registry.py                      # 9 ML models + Optuna HP search spaces
│   │
│   └── training/
│       ├── __init__.py
│       └── cross_validation.py              # Nested CV, simple CV, result saving
│
├── data/
│   ├── raw/
│   │   └── structure_files/                 # CIF files: 7QKO.cif, 7EKI.cif, 6CNJ.cif, 6PV7.cif
│   └── processed/                           # For cleaned/intermediate data files
│
└── results/                                 # Experiment JSON results saved here
```

---

## 12. Next Steps / Roadmap

### Priority 1: Improve Current Pipeline
1. **Get RSA working:** Install mkdssp on Linux/Mac, or generate DSSP files externally and load them. RSA is potentially the most informative structural feature.
2. **Download AlphaFold structures** for the 5 uncovered subunits (CHRNA2, CHRNA5, CHRNA6, CHRNA9, CHRNB3) to cover the remaining 38 mutations.
3. **Run full Optuna HP optimization** (`python scripts/run_experiment.py --all-models`) and compare with baseline.
4. **Clean duplicate mutations** in the database (same mutation reported by different papers).

### Priority 2: Add New Features
5. **Domain annotations:** Add features for which structural domain the mutation is in (ECD, TM1-4, ICD). Mutations in TM2 (pore lining) are particularly important.
6. **Distance to binding site:** How far is the mutation from the ACh binding pocket? Closer = more likely to affect function.
7. **Conservation scores:** Align nAChR sequences across species (human, mouse, rat, zebrafish, etc.) and compute how conserved each position is.

### Priority 3: Advanced Features
8. **ESM embeddings:** Use protein language models (ESM-2 from Meta) to generate per-residue embeddings. These are learned representations from millions of protein sequences and often outperform hand-crafted features.
9. **Collect mouse nAChR mutation data** and implement species transfer experiment (like ENaC did).

### Priority 4: Analysis
10. **Feature ablation studies:** Remove one feature group at a time to measure which features contribute most.
11. **SHAP feature importance:** Visualize which features the model relies on for individual predictions.

---

## 9. Project File Map

```
VEP Nachr/
│
├── NOTES.ipynb                              # This documentation file
├── requirements.txt                         # Python dependencies (pip install -r requirements.txt)
│
├── scripts/
│   └── run_experiment.py                    # Main entry point for running experiments
│
├── vep_nachr/                               # The Python package (all source code)
│   ├── __init__.py                          # Package marker, version info
│   ├── config.py                            # All settings: paths, subunits, labels, model lists
│   │
│   ├── data/
│   │   ├── __init__.py
│   │   └── loader.py                        # Load Excel DB, clean data, load FASTA sequences
│   │
│   ├── features/
│   │   ├── __init__.py
│   │   ├── physicochemical.py               # 24 AAIndex features (from ENaC, unchanged)
│   │   ├── substitution.py                  # BLOSUM62 (from ENaC) + Grantham distance (new)
│   │   ├── structural.py                    # PDB-based features (skeleton, needs PDB files)
│   │   └── encoder.py                       # Combines all features into sklearn transformer
│   │
│   ├── models/
│   │   ├── __init__.py
│   │   └── registry.py                      # 9 ML models + Optuna HP search spaces
│   │
│   └── training/
│       ├── __init__.py
│       └── cross_validation.py              # Nested CV, simple CV, result saving
│
├── data/
│   ├── raw/
│   │   └── structure_files/                 # Put PDB files here (7EKI.pdb, 7QKO.pdb, etc.)
│   └── processed/                           # For cleaned/intermediate data files
│
└── results/                                 # Experiment JSON results saved here
```

---

## 10. Next Steps / Roadmap

### Priority 1: Improve Current Pipeline
1. **Clean duplicate mutations** in the database (same mutation reported by different papers)
2. **Implement structural feature extraction** from the 4 downloaded PDB files (7QKO, 7EKI, 6CNJ, 6PV7)
3. **Run full Optuna HP optimization** and compare with baseline

### Priority 2: Add New Features
4. **Domain annotations:** Add features for which structural domain the mutation is in (ECD, TM1-4, ICD). Mutations in TM2 (pore lining) are particularly important.
5. **Distance to binding site:** How far is the mutation from the ACh binding pocket? Closer = more likely to affect function.
6. **Conservation scores:** Align nAChR sequences across species (human, mouse, rat, zebrafish, etc.) and compute how conserved each position is. Highly conserved positions are more likely to be functionally important.

### Priority 3: More Data
7. **Collect mouse nAChR mutation data** and implement species transfer experiment (like ENaC did)
8. **ESM embeddings:** Use protein language models (ESM-2 from Meta) to generate per-residue embeddings. These are learned representations from millions of protein sequences and often outperform hand-crafted features.

### Priority 4: Analysis
9. **Feature ablation studies:** Remove one feature group at a time to measure which features contribute most
10. **SHAP feature importance:** Visualize which features the model relies on for individual predictions